# 03 Robustness analyses, own cohort (Supplementary Table S2, values in Figure 1H and Results)
Fresh kernel, runs in about two minutes. Needs `data/own_cohort_xenium_247panel.h5ad`.
Writes `tables/Z1` to `Z5`. Z2 feeds Figure 1H (notebook 01) and Supplementary Table S2 (notebook 04).

| Cell | Question |
|---|---|
| Z0 | cell tables: local context, home region, astrocyte neighbours |
| Z1 | is the astrocyte association stronger in the aged brain (age x astrocyte interaction) |
| Z2 | astrocyte profiles next to Spp1+ microglia: contact excluded, radii, homeostatic controls, depth |
| Z3 | astrocyte-poor or oligodendrocyte-rich niche (composition) |
| Z4 | radius, Spp1 threshold, leave one home region out |
| Z5 | coverage of microglia in fibre vs grey tissue |

In [ ]:
# ==========================================================
# Z0) Revision analyses, own cohort (K1): fresh kernel, cell tables with context and home region
# ==========================================================
import numpy as np, pandas as pd, scanpy as sc, gc, warnings
from pathlib import Path
from scipy.spatial import cKDTree
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")

import os
# Project folder with data/, tables/, figures/ and final_data/ (see README).
# Default: the parent of the notebooks folder. Override with the environment variable SPP1_PROJECT_DIR.
OUT = Path(os.environ.get("SPP1_PROJECT_DIR", Path.cwd().parent)).resolve()
FIG, TAB, DAT = OUT / "figures", OUT / "tables", OUT / "data"
if "C" in globals() and not callable(C): del C          # protect patsy's C()

SAMPLES = ["SPF_Coronar", "AQP4KO_sagital", "Thy_CTRL"]
AGED = "Thy_CTRL"
LAB = {"SPF_Coronar": "SPF 12 wk", "AQP4KO_sagital": "AQP4 KO 12 wk", "Thy_CTRL": "Healthy 1 y"}
NOT_HOME = ["White matter", "Ventricle / CP"]
FORE = ["Cortex", "Hippocampus", "Striatum", "Pallidum", "Thalamus", "Hypothalamus"]
RADII = [15, 25, 35, 50]
rng = np.random.default_rng(0)

adata = sc.read_h5ad(DAT / "own_cohort_xenium_247panel.h5ad")
adata = adata[adata.obs["sample"].isin(SAMPLES)].copy()
Xc = adata.layers["counts"].tocsr()
depth = np.asarray(Xc.sum(1)).ravel()
gi = {g: i for i, g in enumerate(adata.var_names)}


def home_region(xy_q, xy_ref, reg_ref, ks=(100, 300, 1000), chunk=20000):
    """Majority grey-matter region among the k nearest cells (fibre tracts and ventricles inherit the region they traverse)."""
    ok = ~pd.Series(reg_ref).isin(NOT_HOME + ["nan"]).to_numpy() & pd.notna(reg_ref)
    cats = np.unique(reg_ref[ok]); code = np.full(len(reg_ref), -1)
    code[ok] = np.searchsorted(cats, reg_ref[ok])
    tree, out, todo = cKDTree(xy_ref), np.full(len(xy_q), -1), np.arange(len(xy_q))
    for k in ks:
        k = min(k, len(xy_ref)); left = []
        for s0 in range(0, len(todo), chunk):
            t = todo[s0:s0 + chunk]
            c = code[tree.query(xy_q[t], k=k)[1]]
            cnt = np.stack([(c == j).sum(1) for j in range(len(cats))], 1)
            hit = cnt.sum(1) > 0
            out[t[hit]] = cnt[hit].argmax(1); left.append(t[~hit])
        todo = np.concatenate(left) if left else np.array([], int)
        if len(todo) == 0: break
    return np.where(out >= 0, cats[np.clip(out, 0, None)], "none")


MG, AS, GEO = [], [], {}
for smp in SAMPLES:
    m = np.where((adata.obs["sample"] == smp).to_numpy())[0]
    xy = np.asarray(adata.obsm["spatial"])[m]
    ct = adata.obs["cell_type"].astype(str).to_numpy()[m]
    reg = adata.obs["region_coarse"].astype(str).to_numpy()[m]
    is_mg, is_as, is_ol = ct == "Microglia", ct == "Astrocyte", ct == "Oligodendrocyte"
    t_all, t_as, t_ol = cKDTree(xy), cKDTree(xy[is_as]), cKDTree(xy[is_ol])
    q = np.where(is_mg | is_as)[0]                                   # microglia and astrocytes
    nb = t_all.query(xy[q], k=31)[1][:, 1:]                           # 30 nearest neighbours, self excluded
    f_ol = is_ol[nb].mean(1)
    home = home_region(xy[q], xy, reg)
    d = pd.DataFrame({"sample": smp, "row": m[q], "x": xy[q, 0], "y_um": xy[q, 1], "ct": ct[q], "region": reg[q],
                      "home": home, "f_oligo": f_ol})
    d["ctx"] = pd.cut(d["f_oligo"], [-0.01, 0.2, 0.4, 1.0], labels=["Grey", "Border", "Fibre"]).astype(str)
    for R in RADII:
        d[f"n_astro{R}"] = t_as.query_ball_point(xy[q], R, return_length=True) - (ct[q] == "Astrocyte")
        d[f"n_oligo{R}"] = t_ol.query_ball_point(xy[q], R, return_length=True)
        d[f"n_total{R}"] = t_all.query_ball_point(xy[q], R, return_length=True) - 1
    d["n_astro_shell"] = d["n_astro25"] - (t_as.query_ball_point(xy[q], 10, return_length=True) - (ct[q] == "Astrocyte"))
    d["spp1_n"] = np.asarray(Xc[m[q], gi["Spp1"]].todense()).ravel()
    MG.append(d[d["ct"] == "Microglia"]); AS.append(d[d["ct"] == "Astrocyte"])
    GEO[smp] = dict(xy=xy, rows=m, is_mg=is_mg, is_as=is_as)

mgz = pd.concat(MG, ignore_index=True); asz = pd.concat(AS, ignore_index=True)
mgz["y"] = (mgz["spp1_n"] >= 1).astype(int); mgz["y2"] = (mgz["spp1_n"] >= 2).astype(int)
mgz["aged"] = (mgz["sample"] == AGED).astype(int)
mgz["any25"] = (mgz["n_astro25"] >= 1).astype(int)
mgz["fb"] = mgz["ctx"].isin(["Border", "Fibre"])
shared = [r for r in FORE if all(((mgz["sample"] == s) & (mgz["home"] == r)).sum() >= 20 for s in SAMPLES)]
mgz["fore_shared"] = mgz["home"].isin(shared)
mgz.to_parquet(DAT / "Z0_microglia_home_ctx.parquet"); asz.to_parquet(DAT / "Z0_astrocytes_home_ctx.parquet")


def glm_or(d, formula, term, strata=("ctx", "home"), y="y"):
    """Binomial GLM, HC1 robust SE. Strata without any positive cell are dropped (avoids separation)."""
    for c in strata:
        if f"C({c})" in formula:
            d = d[d.groupby(c)[y].transform("sum") > 0]
    mod = smf.glm(formula, data=d, family=sm.families.Binomial())
    try:
        f = mod.fit(cov_type="HC1")
    except Exception:
        f = mod.fit(method="bfgs", maxiter=1000, cov_type="HC1")
    lo, hi = np.exp(f.conf_int().loc[term])
    return pd.Series({"OR": np.exp(f.params[term]), "lo": lo, "hi": hi, "p": f.pvalues[term],
                      "n": len(d), "n_pos": int(d[y].sum())})


print("forebrain home regions shared by all three sections:", shared)
print(mgz.groupby("sample").agg(microglia=("y", "size"), spp1_pos=("y", "sum"), pct=("y", "mean")).assign(pct=lambda t: (100 * t.pct).round(1)))
print("\nsanity check vs paper (aged, per astrocyte OR ~0.71):")
print(glm_or(mgz[mgz["aged"] == 1], "y ~ n_astro25 + C(ctx) + C(home)", "n_astro25").round(3).to_string())

In [ ]:
# ==========================================================
# Z1) A1: is the astrocyte association specific to the aged brain? (age x astrocyte interaction)
# ==========================================================
rows = []
SETS = {"all three sections": mgz,
        "same plane (1 y vs AQP4 KO)": mgz[mgz["sample"] != "SPF_Coronar"],
        "border + fibre only": mgz[mgz["fb"]],
        "shared forebrain": mgz[mgz["fore_shared"]]}
for name, d in SETS.items():
    for term_var in ["n_astro25", "any25"]:
        f = f"y ~ aged * {term_var} + C(ctx) + C(home)" if name != "border + fibre only" else f"y ~ aged * {term_var} + C(home)"
        r = glm_or(d, f, f"aged:{term_var}"); r["model"] = "interaction"
        rows.append(r.rename(f"{name} | {term_var}"))
        for smp in d["sample"].unique():                 # slope within each section, same covariates
            dd = d[d["sample"] == smp]
            try:
                r = glm_or(dd, f.replace(f"aged * {term_var}", term_var), term_var); r["model"] = f"slope {LAB[smp]}"
                rows.append(r.rename(f"{name} | {term_var}"))
            except Exception as e:
                print(name, smp, term_var, "failed:", e)
z1 = pd.DataFrame(rows)
z1.round(4).to_csv(TAB / "Z1_age_x_astrocyte_interaction.csv")
print("interaction OR < 1 = the astrocyte association is stronger in the aged brain than in the young references")
display(z1.set_index("model", append=True)[["OR", "lo", "hi", "p", "n_pos"]].round(3))

# contact share in border + fibre context: Fisher per section, and the difference between aged and young as a test
from scipy.stats import fisher_exact
print("\nShare of microglia with >=1 astrocyte within 25 µm, border + fibre context:")
for smp in SAMPLES:
    t = mgz[(mgz["sample"] == smp) & mgz["fb"]]
    tab = pd.crosstab(t["y"], t["any25"]).reindex(index=[1, 0], columns=[1, 0], fill_value=0)
    print(f"  {LAB[smp]:14s} Spp1+ {100 * t.loc[t.y == 1, 'any25'].mean():5.1f}% (n={int(t.y.sum())})  "
          f"Spp1- {100 * t.loc[t.y == 0, 'any25'].mean():5.1f}% (n={int((t.y == 0).sum())})  "
          f"Fisher p = {fisher_exact(tab.to_numpy())[1]:.3f}")

In [ ]:
# ==========================================================
# Z2) A2 + A3: astrocytes next to Spp1+ vs Spp1- microglia: contact shell, radii, homeostatic controls, depth
# ==========================================================
GENES = [g for g in ["Gfap", "Aqp4", "Cd44", "Id2", "Gli3",                    # reactive / candidate genes
                     "Acsbg1", "Slc39a12", "Ntsr2", "Rfx4"] if g in gi]          # homeostatic astrocyte controls
CONTROLS = ["Acsbg1", "Slc39a12", "Ntsr2", "Rfx4"]
CONFIGS = [("0-25 µm", 0, 25), ("contact excluded 10-25 µm", 10, 25), ("0-15 µm", 0, 15), ("0-35 µm", 0, 35)]
N_PERM = 10000


def astro_niche(smp, r_min, R, excl=50, min_ref=10):
    """Astrocytes within [r_min, R) of a Spp1+ microglia vs astrocytes within [r_min, R) of Spp1- microglia only
    (and > excl µm from any Spp1+), compared within strata of context x home region."""
    mg_s = mgz[mgz["sample"] == smp]; as_s = asz[asz["sample"] == smp]
    pos = mg_s["y"].to_numpy() == 1
    xy_as = as_s[["x", "y_um"]].to_numpy()
    d_pos = cKDTree(mg_s.loc[pos, ["x", "y_um"]].to_numpy()).query(xy_as)[0]
    d_neg = cKDTree(mg_s.loc[~pos, ["x", "y_um"]].to_numpy()).query(xy_as)[0]
    near_pos = (d_pos >= r_min) & (d_pos < R)
    near_neg = (d_neg >= r_min) & (d_neg < R) & (d_pos > excl)
    sel = near_pos | near_neg
    rows_ = as_s["row"].to_numpy()[sel]
    lab = near_pos[sel]
    st = (as_s["ctx"] + "|" + as_s["home"]).to_numpy()[sel]
    keep_st = [s for s in np.unique(st) if (lab & (st == s)).sum() > 0 and (~lab & (st == s)).sum() >= min_ref]
    k = np.isin(st, keep_st); rows_, lab, st = rows_[k], lab[k], st[k]
    cnt = Xc[rows_][:, [gi[g] for g in GENES]].toarray().astype(float)
    det = (cnt > 0).astype(float)
    expr = np.log1p(cnt / depth[rows_, None] * 100)
    idx = {s: np.where(st == s)[0] for s in keep_st}

    def oe(L, M):
        o = M[L].mean(0); e = np.zeros(M.shape[1])
        for ii in idx.values():
            Lr = L[ii]; n = Lr.sum()
            if n: e += n * M[ii][~Lr].mean(0)
        return o, e / L.sum()

    od, ed = oe(lab, det); oxp, exx = oe(lab, expr)
    dep_o, dep_e = oe(lab, np.log10(depth[rows_])[:, None])
    nd, nx = np.zeros((N_PERM, len(GENES))), np.zeros((N_PERM, len(GENES)))
    for i in range(N_PERM):
        lp = lab.copy()
        for ii in idx.values(): lp[ii] = rng.permutation(lp[ii])
        a, b = oe(lp, det); nd[i] = a - b
        a, b = oe(lp, expr); nx[i] = a - b
    p_det = (np.sum(np.abs(nd) >= np.abs(od - ed), 0) + 1) / (N_PERM + 1)
    p_exp = (np.sum(np.abs(nx) >= np.abs(oxp - exx), 0) + 1) / (N_PERM + 1)
    return pd.DataFrame({"gene": GENES, "n_near_pos": int(lab.sum()), "n_ref": int((~lab).sum()),
                         "det_near_pos": 100 * od, "det_expected": 100 * ed, "p_det": p_det,
                         "expr_near_pos": oxp, "expr_expected": exx,
                         "fold_expr": np.expm1(oxp) / np.expm1(exx), "p_expr": p_exp,
                         "depth_ratio": 10 ** (dep_o[0] - dep_e[0])})


z2 = []
for smp in SAMPLES:
    for name, rmin, R in (CONFIGS if smp == AGED else CONFIGS[:2]):
        r = astro_niche(smp, rmin, R)
        r["q_expr"] = multipletests(r["p_expr"], method="fdr_bh")[1]
        r.insert(0, "config", name); r.insert(0, "sample", LAB[smp]); z2.append(r)
z2 = pd.concat(z2, ignore_index=True)
z2.round(4).to_csv(TAB / "Z2_astro_near_spp1_shell_radius_controls.csv", index=False)

print("fold_expr = expression in astrocytes next to Spp1+ / expected from astrocytes next to Spp1- in the same strata")
display(z2.pivot_table(index="gene", columns=["sample", "config"], values="fold_expr", sort=False).round(2))
display(z2.pivot_table(index="gene", columns=["sample", "config"], values="p_expr", sort=False).round(3))
print("\nsequencing depth of astrocytes next to Spp1+ relative to expected (1.0 = no difference):")
display(z2.groupby(["sample", "config"], sort=False)[["n_near_pos", "n_ref", "depth_ratio"]].first().round(2))

In [ ]:
# ==========================================================
# Z3) A4: is the niche astrocyte-poor or just oligodendrocyte-rich? (neighbourhood composition, 25 and 50 µm)
# ==========================================================
N_PERM3 = 5000


def composition(d, R):
    n_tot = d[f"n_total{R}"].to_numpy(float); n_as = d[f"n_astro{R}"].to_numpy(float); n_ol = d[f"n_oligo{R}"].to_numpy(float)
    with np.errstate(invalid="ignore", divide="ignore"):
        return pd.DataFrame({"cells (density)": n_tot,
                             "astrocyte share": np.where(n_tot > 0, n_as / n_tot, np.nan),
                             "astrocyte share, oligodendrocytes removed": np.where(n_tot - n_ol > 0, n_as / (n_tot - n_ol), np.nan),
                             "oligodendrocyte share": np.where(n_tot > 0, n_ol / n_tot, np.nan)})


def strat_ratio(d, R, strata_cols):
    """Mean in Spp1+ microglia / expected mean from Spp1- microglia of the same strata, permutation within strata."""
    M = composition(d, R).to_numpy(); lab = d["y"].to_numpy() == 1
    st = d[strata_cols].astype(str).agg("|".join, axis=1).to_numpy()
    keep = [s for s in np.unique(st) if (lab & (st == s)).sum() > 0 and (~lab & (st == s)).sum() >= 10]
    k = np.isin(st, keep); M, lab, st = M[k], lab[k], st[k]
    idx = [np.where(st == s)[0] for s in keep]

    def oe(L):
        o = np.nanmean(M[L], 0); e = np.zeros(M.shape[1])
        for ii in idx:
            n = L[ii].sum()
            if n: e += n * np.nanmean(M[ii][~L[ii]], 0)
        return o, e / L.sum()

    o, e = oe(lab); null = np.zeros((N_PERM3, M.shape[1]))
    for i in range(N_PERM3):
        lp = lab.copy()
        for ii in idx: lp[ii] = rng.permutation(lp[ii])
        a, b = oe(lp); null[i] = a / b
    p = (np.sum(np.abs(np.log(null)) >= np.abs(np.log(o / e)), 0) + 1) / (N_PERM3 + 1)
    return pd.DataFrame({"metric": composition(d.iloc[:1], R).columns, "R": R, "n_pos": int(lab.sum()),
                         "Spp1+": o, "expected": e, "ratio": o / e, "p": p})


z3 = []
for name, d, cols in [("pooled, three sections", mgz, ["sample", "ctx", "home"])] + \
                     [(LAB[s], mgz[mgz["sample"] == s], ["ctx", "home"]) for s in SAMPLES]:
    for R in [25, 50]:
        r = strat_ratio(d, R, cols); r.insert(0, "set", name); z3.append(r)
z3 = pd.concat(z3, ignore_index=True)
z3.round(4).to_csv(TAB / "Z3_niche_composition.csv", index=False)
print("ratio < 1: lower around Spp1+ microglia than around Spp1- microglia of the same context and home region")
display(z3.pivot_table(index=["set", "R"], columns="metric", values="ratio", sort=False).round(2))
display(z3.pivot_table(index=["set", "R"], columns="metric", values="p", sort=False).round(3))

In [ ]:
# ==========================================================
# Z4) A5: sensitivity grid for Supplementary Table S1 (1-year brain): radius x Spp1 threshold x scope, leave-one-region-out
# ==========================================================
aged_mg = mgz[mgz["aged"] == 1].copy()
grid = []
for scope, d in [("whole section", aged_mg), ("border + fibre", aged_mg[aged_mg["fb"]]),
                 ("shared forebrain", aged_mg[aged_mg["fore_shared"]])]:
    for R in [15, 25, 35]:
        for yv, thr in [("y", "≥1"), ("y2", "≥2")]:
            f = f"{yv} ~ n_astro{R} + C(home)" + ("" if scope == "border + fibre" else " + C(ctx)")
            try:
                r = glm_or(d, f, f"n_astro{R}", y=yv)
            except Exception as e:
                print(scope, R, thr, "failed:", e); continue
            grid.append(r.rename((scope, R, thr)))
grid = pd.DataFrame(grid); grid.index.names = ["scope", "radius µm", "Spp1 transcripts"]

loro = []
for reg in sorted(aged_mg["home"].unique()):
    d = aged_mg[aged_mg["home"] != reg]
    if d["y"].sum() < 20: continue
    r = glm_or(d, "y ~ n_astro25 + C(ctx) + C(home)", "n_astro25"); loro.append(r.rename(f"without {reg}"))
loro = pd.DataFrame(loro)

grid.round(4).to_csv(TAB / "Z4_S1_sensitivity_grid.csv"); loro.round(4).to_csv(TAB / "Z4_S1_leave_one_region_out.csv")
print("OR per astrocyte neighbour (1-year brain), context and home region as fixed effects")
display(grid[["OR", "lo", "hi", "p", "n_pos"]].round(3))
print("\nleave one home region out (25 µm, ≥1 transcript):")
display(loro[["OR", "lo", "hi", "p", "n_pos"]].round(3))
print(f"range OR {loro['OR'].min():.2f} to {loro['OR'].max():.2f}, max p = {loro['p'].max():.3f}")

In [ ]:
# ==========================================================
# Z5) A6: are microglia in fibre-rich tissue less covered by astrocytes, already in young brains?
# ==========================================================
cov = (mgz.assign(ctx=pd.Categorical(mgz["ctx"], ["Grey", "Border", "Fibre"]))
          .groupby(["sample", "ctx"], observed=True)
          .agg(microglia=("y", "size"), astro_per_mg=("n_astro25", "mean"), any_astro_pct=("any25", "mean")))
cov["any_astro_pct"] *= 100
# astrocyte share of all cells per context (is fibre tissue simply poorer in astrocytes?)
cells = []
for smp in SAMPLES:
    a = adata[adata.obs["sample"] == smp]
    xy = np.asarray(a.obsm["spatial"]); ct = a.obs["cell_type"].astype(str).to_numpy()
    nb = cKDTree(xy).query(xy, k=31)[1][:, 1:]
    ctx_all = pd.cut((ct == "Oligodendrocyte")[nb].mean(1), [-0.01, 0.2, 0.4, 1.0], labels=["Grey", "Border", "Fibre"])
    cells.append(pd.DataFrame({"sample": smp, "ctx": ctx_all, "astro": ct == "Astrocyte"}))
cells = pd.concat(cells)
cov["astro_share_of_cells_pct"] = 100 * cells.groupby(["sample", "ctx"], observed=True)["astro"].mean()
cov = cov.rename(index=LAB, level=0)
display(cov.round(2))

rows = []
for smp in SAMPLES:
    d = mgz[(mgz["sample"] == smp) & (mgz["y"] == 0)].copy()          # Spp1- microglia only: baseline coverage
    d["fibre"] = d["fb"].astype(int)
    for var, fam in [("any25", "contact"), ("n_astro25", "count")]:
        if fam == "contact":
            r = glm_or(d.rename(columns={"any25": "yy"}), "yy ~ fibre + C(home)", "fibre", strata=("home",), y="yy")
        else:
            f = smf.glm("n_astro25 ~ fibre + C(home)", data=d, family=sm.families.Poisson()).fit(cov_type="HC1")
            lo, hi = np.exp(f.conf_int().loc["fibre"])
            r = pd.Series({"OR": np.exp(f.params["fibre"]), "lo": lo, "hi": hi, "p": f.pvalues["fibre"], "n": len(d)})
        r["measure"] = "OR any astrocyte" if fam == "contact" else "rate ratio astrocytes"
        rows.append(r.rename(LAB[smp]))
z5 = pd.DataFrame(rows)
z5.round(4).to_csv(TAB / "Z5_astro_coverage_fibre_vs_grey.csv")
print("\nSpp1- microglia, border + fibre vs grey context within the same home region (< 1 = less astrocyte coverage):")
display(z5[["measure", "OR", "lo", "hi", "p", "n"]].round(3))